# RecipeAnalysis - Análisis de Recetas y Calificaciones

Este notebook se conecta a la base de datos MongoDB del proyecto `RecipeManager`, carga las colecciones relacionales (recetas, usuarios y reseñas) en dataframes de **Pandas** y realiza un análisis de popularidad e ingredientes, generando gráficos analíticos con **Seaborn**.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient

# Configurar paths para poder importar el DAO y modelos locales
sys.path.append(os.path.abspath('../app'))
sys.path.append(os.path.abspath('..'))

from app.dao import RecipeManagerDAO
from app.db_models import Recipe, User, Review

In [ ]:
# Conectar a MongoDB
MONGO_URI = os.environ.get('MONGO_URI', 'mongodb://localhost:27017/recipe_db')
client = MongoClient(MONGO_URI)
db = client['recipe_db']
dao = RecipeManagerDAO(db)

# Cargar datos utilizando el DAO
recipes_raw = [r.to_dict() | {"id": r.id} for r in dao.list_recipes()]
users_raw = [u.to_dict() | {"id": u.id} for u in dao.list_users()]

# Para reviews, listamos por cada receta y unimos
reviews_raw = []
for r in recipes_raw:
    for rev in dao.list_reviews_by_recipe(r['id']):
        reviews_raw.append(rev.to_dict() | {"id": rev.id})

# Crear DataFrames
df_recipes = pd.DataFrame(recipes_raw)
df_users = pd.DataFrame(users_raw)
df_reviews = pd.DataFrame(reviews_raw)

print(f"Cargados con éxito: {len(df_recipes)} recetas, {len(df_users)} usuarios y {len(df_reviews)} reseñas.")

## 1. Análisis de Calificaciones Promedio
Unimos las reseñas con los títulos de las recetas y los nombres de los chefs para ver cuáles son las mejor valoradas.

In [ ]:
# Combinar DataFrames para obtener datos cruzados
df_merged = df_reviews.merge(df_recipes, left_on='recipe_id', right_on='id', suffixes=('_review', '_recipe'))
df_merged = df_merged.merge(df_users, left_on='user_id', right_on='id')

# Calcular el promedio de valoración por receta
df_avg_ratings = df_merged.groupby('title')['rating'].mean().reset_index().sort_values(by='rating', ascending=False)
print("Calificaciones promedio de las recetas:")
print(df_avg_ratings.to_string(index=False))

In [ ]:
# Configurar estilo estético moderno
sns.set_theme(style="darkgrid")
plt.figure(figsize=(10, 5))

ax = sns.barplot(x='rating', y='title', data=df_avg_ratings, palette='viridis', hue='title', legend=False)
plt.title('Calificación Promedio por Receta (Escala 1-5 ★)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Valoración Promedio', fontsize=12)
plt.ylabel('Receta', fontsize=12)
plt.xlim(1, 5)

# Añadir etiquetas en las barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f ★', padding=5)

plt.tight_layout()
plt.show()

## 2. Análisis de Ingredientes Más Comunes
Explosionamos la columna de ingredientes de las recetas para contar la frecuencia con la que se usa cada ingrediente.

In [ ]:
# Explotar la lista de ingredientes
df_ingredients = df_recipes.explode('ingredients')
ingredient_counts = df_ingredients['ingredients'].value_counts().reset_index()
ingredient_counts.columns = ['Ingrediente', 'Frecuencia']

print("Ingredientes más populares:")
print(ingredient_counts.head(10).to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x='Frecuencia', y='Ingrediente', data=ingredient_counts.head(12), palette='magma', hue='Ingrediente', legend=False)
plt.title('Ingredientes Más Frecuentes en el Catálogo', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Cantidad de Recetas que lo Utilizan', fontsize=12)
plt.ylabel('Ingrediente', fontsize=12)
plt.xticks(range(0, int(ingredient_counts['Frecuencia'].max()) + 2))
plt.tight_layout()
plt.show()